# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Type: scoring for ranking, built on a binary classifier.**

The deliverable a reviewer actually uses is a **ranked queue** — "look at these pages first." That
output is a ranking problem: order, not category, is what matters to the person acting on it.

Underneath the ranking sits a **binary classification** signal: `is_declining_label` (1 if the page's
search demand is trending down, 0 otherwise). The model's predicted probability of decline is the
score used to sort the queue. So the task type is "binary classification whose output is consumed as
a rank," not classification for its own sake — nobody wants a yes/no, they want an ordered list.

It is **not** clustering (I have a defined outcome to predict, not unlabeled groups to discover), and
it is not the raw regression of a traffic number (the reference pipeline avoids predicting `trend_pct`
directly — that number is noisy and the label is derived from it, which would be circular).

In [ ]:
LANE = "refresh_content_opportunity_scoring"
TASK_TYPE = "binary_classification_consumed_as_ranking"
print(f"lane: {LANE}")
print(f"task type: {TASK_TYPE}")

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**It's a proxy, not an observed outcome.**

I predict `is_declining_label`, defined as `trend_direction == "down"`, itself computed from
`(impressions_last_30d - impressions_prev_30d) / impressions_prev_30d < -20%`. That's a **rule
applied to real search data**, not something a human labeled by reviewing the page.

Two honest limits this creates:

1. **It's a proxy for "should be reviewed," not the thing itself.** A page can be declining in
   impressions for reasons a refresh won't fix (seasonality, a SERP feature stealing the click, a
   competitor outranking it). The model predicts "demand looks like it's dropping," not "editing
   this page will help."
2. **Leakage risk is direct, not hypothetical.** `trend_direction` and `trend_pct` are the *source*
   of the label, so they can never be features — that would let the model "predict" the label from
   itself. `docs/data-dictionary.md` flags this explicitly and it's the exact mistake notebook 02
   demonstrates.

I did not predict `trend_pct` as a regression target for the same reason: predicting the noisy
number that literally *defines* the label is circular, and a 90-day window has enough variance that
a point estimate would be harder to defend than a directional call.

In [ ]:
import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

is_declining_label = (df["trend_direction"] == "down").astype(int)
print("rows:", len(df))
print("declining-label rows:", int(is_declining_label.sum()))
print(f"declining-label rate: {is_declining_label.mean():.3f}")
print()
print("trend_direction breakdown:")
print(df["trend_direction"].value_counts())

## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Precision@50** — of the top 50 pages the model puts at the front of the queue, what fraction are
actually declining?

Why this, not accuracy or ROC AUC: a reviewer only ever works through a fixed number of pages in a
sitting — the top of the queue is the entire product. A model can have a great AUC and still waste
a reviewer's morning if its top 50 aren't good picks. Precision@K measures exactly the thing the
human experiences.

**What "good" means, from the reference pipeline's own committed report
(`outputs/model_report.md`, run on this same dataset):**

| Model | Precision@50 |
|---|---:|
| Hand-written rule baseline | 0.240 |
| Random forest (best model) | 0.740 |

So "good" for my own model means beating the 0.240 baseline by a meaningful margin — the shipped
reference model gets ~3x. I'll treat the baseline, not zero, as the floor to beat, since a model
that ties a simple rule isn't earning the extra complexity.

I'm not using raw accuracy: with the label at 54.2% positive (see cell below), a model that guesses
the majority class already looks "accurate" while being useless for ranking.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("../..").resolve()))
from scripts.ml_utils import precision_at_k

# avg_position == 0 means "no position data" (docs/data-dictionary.md), not the best
# rank -- scoring those rows as elite would be a real bug, so they're excluded here
# rather than filled with a value that fakes a great position.
has_position = df["avg_position"] > 0
scored = df.loc[has_position]
scores = -scored["avg_position"]  # lower avg_position = better = higher score
label_subset = is_declining_label.loc[has_position]

p_at_50 = precision_at_k(label_subset, scores, k=50)
print(f"rows with no position data (excluded): {(~has_position).sum()}")
print(f"Precision@50 using avg_position alone as the score: {p_at_50:.3f}")
print()
print("Committed reference numbers (outputs/model_report.md), for comparison:")
print("  baseline_rules   Precision@50 = 0.240")
print("  random_forest    Precision@50 = 0.740  <- best model, ~3x the baseline")

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**One row = one pseudonymized content page** (`content_id`), not a client and not a day. The full
release (`fact_content_daily_performance`) is grain report_date × client × content, but this lane's
decision — "which pages to look at this week" — is made at the page level, so I collapse to one row
per page before scoring anything.

`client_id` only matters for the **train/test split**: pages from the same client can't leak across
train and test, or the model gets credit for memorizing a client's baseline traffic level instead
of learning the general pattern.

In [ ]:
print("shape (rows, cols):", df.shape)
print("unique content_id:", df["content_id"].nunique(), "(one row per page, confirmed)")
print("unique client_id:", df["client_id"].nunique())
print()
display(
    df[
        [
            "content_id",
            "client_id",
            "content_type",
            "impressions_90d",
            "avg_position",
            "trend_direction",
        ]
    ].head(5)
)

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

There already **is** a fixed rule in this repo (`scripts/02_baseline_score.py`) — I'm not arguing
ML from scratch, I'm arguing it against a real, working baseline.

**Why the rule alone isn't enough:** decline shows up from different combinations of signals —
low CTR with high impressions, good position with falling engagement, high AI-referred traffic
with dropping clicks — and a hand-written rule has to pick fixed thresholds and fixed weights for
each of these ahead of time. The committed comparison shows the cost of that: the rule baseline
gets Precision@50 = 0.240, the random forest gets 0.740. That gap is the rule missing interactions
between signals that it wasn't specifically written to catch.

**Why not skip the rule entirely:** the baseline is still the floor and the sanity check — if my
model can't clear 0.240, something is broken (usually leakage making the rule look artificially
weak by comparison, or a data bug). It's also transparent in a way the model isn't, which matters
if a reviewer wants a reason for a specific page.

The below shows two of the top drivers the model actually leans on
(`outputs/model_report.md` → Top Features), which line up with signals a simple rule treats as
independent but that interact in practice.

In [ ]:
import numpy as np

# Two top features from the committed model report, checked for a simple
# correlation on this dataset (a rule that only thresholds one of these
# separately would miss how they move together).
pair = df[["days_with_impressions", "log_clicks_90d" if "log_clicks_90d" in df.columns else "clicks_90d"]].copy()
if "log_clicks_90d" not in df.columns:
    pair["log_clicks_90d"] = np.log1p(df["clicks_90d"])
    pair = pair.drop(columns=["clicks_90d"])

corr = pair["days_with_impressions"].corr(pair["log_clicks_90d"])
print(f"correlation(days_with_impressions, log_clicks_90d): {corr:.3f}")
print("Top model features (from outputs/model_report.md):")
print("  1. days_with_impressions   0.158")
print("  2. log_impressions_90d     0.128")
print("  3. avg_position            0.109")

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.